# Counts
* total number of recordings per site and year
* total number of minutes per site and year
* total number of inference windows per site and year

## Connect to database
Follow the instruction in the Notebook [00_Setup_DB_Connection](./00_Setup_DB_Connection.ipynb).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Load the database username and password
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read the username and password
username = os.getenv("DB_USERNAME")
password = os.getenv("DB_PASSWORD")

DB_CONFIG = {
    'host': 'localhost',
    'port': 8432,  # the port opened using the ssh tunnel
    'user': username,
    'password': password,
    'database': 'ecomon'
}

# settings for this instance
instance_name = "validate"
model_name = 'BNP_v3_0_glob11k_prev3_cw5s_sz2s'
record_regime_recording_duration = 60
window_length = 5
slide_length = 2

## Test the connection

In [ ]:
from bslquery import BioSoundsLabDB

# Initialize the database helper
db = BioSoundsLabDB(DB_CONFIG, verbose=True)

# Test the connection
db.test_connection()

## Fetch sites, years, recordings count
This will return one entry for each combination of site prefix and year, and the count of records.

In [ ]:
import pandas as pd

# List all sites and years combinations
site_years = db.sites_years()
site_years = pd.DataFrame(site_years)
site_years.rename(columns = {'count': 'recordings'}, inplace=True)

Add total number of seconds recorded. This is an approximation based on the recording regime recording duration for this site.

In [ ]:
site_years['seconds'] = site_years['recordings'] * record_regime_recording_duration

Add total number of windows, based on recording length, window size and slide length.
For this specific case, the window will not slide past the end of the recording.

In [ ]:
window_num = ((record_regime_recording_duration - window_length) // slide_length) + 1
site_years['windows'] = site_years['recordings'] * window_num

In [ ]:
site_years.head()

___Before executing the next cell, create a folder where the spreadsheets with the data will be stored.___

The folder should be called: examples/data (this folder will not be committed). 

In [ ]:
# export as excel
filename = f"Falco_Counts_{instance_name}.xlsx".replace(' ', '_')
site_years.to_excel(
    f"data/{filename}",
    sheet_name='Data',
    index=False,
    header=True,
)